# Lab 17-04 · Pulses, Eyes and Equalizers

Five parts, each built on `baseband.py`. **Predict first**, then run. The handout is [README.md](README.md); Part 6 (ngspice) runs outside the notebook.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import baseband as bb
rng = np.random.default_rng(1704)

## Part 1 · Nyquist's criterion

*Predict:* which of the four spectra of Problem 17-04.5 give zero ISI.

In [2]:
specs = {
    "a: triangle to 1/T": lambda q: np.clip(1 - np.abs(q), 0, None),
    "b: flat to 0.6/T": lambda q: (np.abs(q) < 0.6).astype(float),
    "c: flat to 0.5/T": lambda q: (np.abs(q) < 0.5).astype(float),
    "d: trapezoid 0.3-0.7/T": lambda q: np.clip((0.7 - np.abs(q)) / 0.4, 0, 1),
}
ff = np.linspace(-0.49, 0.49, 999)
fg = np.linspace(-1, 1, 200001); dfg = fg[1] - fg[0]
for name, H in specs.items():
    dev = np.max(np.abs(bb.folded_spectrum(H, ff) - 1))
    h = lambda t: np.sum(H(fg) * np.cos(2 * np.pi * fg * t)) * dfg
    print(f"{name:24s}: folded-spectrum deviation {dev:.3f}, h(T)/h(0) = {h(1.0) / h(0.0):+.4f}")

a: triangle to 1/T      : folded-spectrum deviation 0.000, h(T)/h(0) = +0.0000
b: flat to 0.6/T        : folded-spectrum deviation 1.000, h(T)/h(0) = -0.1559
c: flat to 0.5/T        : folded-spectrum deviation 0.000, h(T)/h(0) = +0.0000
d: trapezoid 0.3-0.7/T  : folded-spectrum deviation 0.000, h(T)/h(0) = +0.0000


## Part 2 · Eyes on a one-pole channel

*Predict:* the worst-case eye $1 - 2r$ at 10, 25 and 45 Gb/s through a 5 GHz pole.

In [3]:
tau = 1 / (2 * math.pi * 5e9)
sps = 64
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, rate in zip(axes, (10e9, 25e9, 45e9)):
    T = 1 / rate
    a = rng.choice([-1.0, 1.0], 4000)
    y = bb.rc_channel(bb.waveform(a, sps), tau / (T / sps))
    samples = y[sps - 1::sps][50:]           # end of each bit
    meas = np.min(samples * a[50:])          # smallest signed distance from the threshold
    r = math.exp(-T / tau)
    print(f"{rate / 1e9:4.0f} Gb/s: predicted worst eye {1 - 2 * r:.4f}, smallest sample in 4000 random bits {meas:.4f}")
    assert meas >= 1 - 2 * r - 1e-9
    for row in bb.fold_eye(y[sps * 20:], sps, span=2)[:200]:
        ax.plot(np.arange(1, 2 * sps + 1) / sps, row, lw=0.3, alpha=0.4)
    ax.set_title(f"{rate / 1e9:.0f} Gb/s")
plt.show()

  10 Gb/s: predicted worst eye 0.9136, smallest sample in 4000 random bits 0.9136


  25 Gb/s: predicted worst eye 0.4308, smallest sample in 4000 random bits 0.4308
  45 Gb/s: predicted worst eye 0.0050, smallest sample in 4000 random bits 0.0051


## Part 3 · Timing sensitivity

*Predict:* how quickly the worst-case eye of a sinc pulse closes as the sampling instant moves, compared with raised cosines.

In [4]:
k = np.arange(-200, 201); k = k[k != 0]
eps = np.linspace(0, 0.3, 31)
for name, fn in (("sinc", np.sinc), ("RC 0.22", lambda t: bb.raised_cosine(t, 0.22)), ("RC 0.5", lambda t: bb.raised_cosine(t, 0.5))):
    eye = [float(fn(np.array([e]))[0] - np.sum(np.abs(fn(k + e)))) for e in eps]
    plt.plot(eps, eye, label=name)
    print(f"{name:8s}: eye at 0.1T {eye[10]:+.3f}, at 0.25T {eye[25]:+.3f}")
plt.axhline(0, color="k", lw=0.5); plt.xlabel("timing error / T"); plt.ylabel("worst-case eye"); plt.legend(); plt.show()

sinc    : eye at 0.1T -0.175, at 0.25T -1.781
RC 0.22 : eye at 0.1T +0.627, at 0.25T +0.052
RC 0.5  : eye at 0.1T +0.785, at 0.25T +0.400


## Part 4 · Equalization

*Predict:* the bit-error rate at σ = 0.25 for the channel of Example 17-04.5 with no equalizer, the three-tap ZF FFE and a two-tap DFE.

In [5]:
h = np.array([0.1, 1.0, 0.4, 0.15])
c = bb.zf_equalizer(h, 1, 3)
print("ZF taps", np.round(c, 4), " combined response", np.round(bb.apply_ffe(h, c), 4))
sigma, n = 0.25, 1_000_000
raw = bb.simulate_binary(h, 1, sigma, n, rng)
ffe = bb.simulate_binary(h, 1, sigma, n, rng, ffe=c)
dfe = bb.simulate_binary(h, 1, sigma, n, rng, dfe=[0.4, 0.15])
print(f"no equalizer {raw:.2e} (statistical eye {bb.statistical_ber(h, 1, sigma):.2e})")
print(f"ZF FFE       {ffe:.2e}")
print(f"2-tap DFE    {dfe:.2e} (statistical eye with only the precursor {bb.statistical_ber([0.1, 1.0], 1, sigma):.2e})")
assert dfe < raw and ffe < raw

ZF taps [-0.1085  1.0852 -0.4178]  combined response [-0.0109  0.      1.      0.     -0.0043 -0.0627]


no equalizer 1.25e-02 (statistical eye 1.25e-02)
ZF FFE       3.77e-04
2-tap DFE    7.70e-05 (statistical eye with only the precursor 8.23e-05)


## Part 5 · PAM4 against NRZ on a one-pole channel

*Predict:* at 25 Gb/s through the 5 GHz pole, which has the larger worst-case eye as a fraction of the common peak swing (equal transmitter swing)?

In [6]:
def worst_eye_pam(m, rate_bits):
    T = math.log2(m) / rate_bits
    r = math.exp(-T / tau)
    hk = bb.rc_cursors(r, 60)
    lv = bb.pam_levels(m)
    spacing = 2.0
    # worst inner-eye opening: main cursor times half the spacing minus the largest ISI excursion
    isi = (m - 1) * (np.sum(np.abs(hk)) - hk[0])
    return (hk[0] * spacing / 2 - isi) / (m - 1)   # normalized to the peak level
for rate in (10e9, 25e9, 40e9):
    print(f"{rate / 1e9:4.0f} Gb/s: NRZ worst eye {worst_eye_pam(2, rate):+.3f}, PAM4 worst eye {worst_eye_pam(4, rate):+.3f} (both as a fraction of the peak level)")

  10 Gb/s: NRZ worst eye +0.914, PAM4 worst eye +0.331 (both as a fraction of the peak level)
  25 Gb/s: NRZ worst eye +0.431, PAM4 worst eye +0.225 (both as a fraction of the peak level)
  40 Gb/s: NRZ worst eye +0.088, PAM4 worst eye +0.056 (both as a fraction of the peak level)


## Part 6 · ngspice

Run `python tools/run_spice.py 17-04` from the repository root: the one-pole channel's cursors at the bit ends are 0.715 and 0.204.